# Feature Creation Notebook

Authors: Harry Zhang, Thomas Saleh, Elena Milito
Packages authored by: Idil Yaktubay, Amanbir Singh, Harry Zhang, Thomas Saleh 

CODE PURPOSE: Create feature tables for total dissolved solids, turbidity, total organic carbon, and discharge at Pinawa and Whitemouth AquaHive stations, for the purpose of parameter forecasting

## Set-up
Import packages

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import os
import sys
import feature_creation as fc
sys.path.append('..')



Import ERA5 met timeseries data

In [3]:
full_dataset = pd.read_csv("met_ts_t2m_cdir_CR.csv")

## Generate Timelag features

### Cypress River Temperature

Calculating time-lag features from temperature dataset

In [4]:
t2m_cr_1_12_features = fc.calc_param_site_features_hrs_ago(df=full_dataset, 
                                                        param='t2m',
                                                        site='cr', 
                                                        hours_ago_start=1, 
                                                        hours_ago_end=3)

t2m_cr_13_48_features = fc.calc_param_site_features_hrs_ago(df=full_dataset, 
                                                        param='t2m',
                                                        site='cr', 
                                                        hours_ago_start=4, 
                                                        hours_ago_end=6)

t2m_cr_49_120_features = fc.calc_param_site_features_hrs_ago(df=full_dataset, 
                                                        param='t2m',
                                                        site='cr', 
                                                        hours_ago_start=7, 
                                                        hours_ago_end=9)

t2m_cr_121_720_features = fc.calc_param_site_features_hrs_ago(df=full_dataset, 
                                                        param='t2m',
                                                        site='cr', 
                                                        hours_ago_start=10, 
                                                        hours_ago_end=12)


In [5]:
full_dataset.info

<bound method DataFrame.info of                   timestamp  discharge parameter          value site
0      2016-01-01T00:00:00Z        0.0       t2m     265.051697   cr
1      2016-01-01T00:00:00Z        0.0      cdir       0.000000   cr
2      2016-01-01T01:00:00Z        0.0       t2m     265.200623   cr
3      2016-01-01T01:00:00Z        0.0      cdir       0.000000   cr
4      2016-01-01T02:00:00Z        0.0       t2m     265.428375   cr
...                     ...        ...       ...            ...  ...
87691  2020-12-31T21:00:00Z        0.0      cdir  567466.812500   cr
87692  2020-12-31T22:00:00Z        0.0       t2m     265.710480   cr
87693  2020-12-31T22:00:00Z        0.0      cdir  268331.875000   cr
87694  2020-12-31T23:00:00Z        0.0       t2m     265.873047   cr
87695  2020-12-31T23:00:00Z        0.0      cdir   31343.511720   cr

[87696 rows x 5 columns]>

Combine temperature features from different time windows into a single table

In [6]:
t2m_full_data = full_dataset[full_dataset['parameter'] == 't2m'].iloc[:, :4]
t2m_full_data['timestamp_ccentral'] = pd.to_datetime(full_dataset['timestamp'], utc=True).dt.tz_convert('Canada/Central')
t2m_full_data.drop('timestamp', axis=1, inplace=True)


# Create final turbidity concentration feature table (wmth)
t2m_cr_final_features_conc = pd.concat([t2m_full_data.set_index('timestamp_ccentral'),
                                        t2m_cr_1_12_features[0].set_index('timestamp_ccentral'),
                                        t2m_cr_13_48_features[0].set_index('timestamp_ccentral'),
                                        t2m_cr_49_120_features[0].set_index('timestamp_ccentral'),
                                        t2m_cr_121_720_features[0].set_index('timestamp_ccentral')
                                        ],
                                        axis=1)
# Create final turbidity load feature table (wmth)
t2m_cr_final_features_load = pd.concat([t2m_cr_1_12_features[1].set_index('timestamp_ccentral'),
                                          t2m_cr_13_48_features[1].set_index('timestamp_ccentral'),
                                          t2m_cr_49_120_features[1].set_index('timestamp_ccentral'),
                                          t2m_cr_121_720_features[1].set_index('timestamp_ccentral')
                                          ],
                                          axis=1)

Reviewing temperature features dataset

In [7]:
fc.feature_checks(t2m_cr_final_features_conc)
t2m_cr_final_features_conc.head()

columns Index(['discharge', 'parameter', 'value', 't2m_cr_mean_1_3', 't2m_cr_mean_4_6',
       't2m_cr_mean_7_9', 't2m_cr_mean_10_12'],
      dtype='object')
start time 2015-12-31 19:00:00-06:00
end time 2020-12-31 17:00:00-06:00
number of rows 43848 and columns 7.000000
Count of non-null values by column


,discharge,parameter,value,t2m_cr_mean_1_3,t2m_cr_mean_4_6,t2m_cr_mean_7_9,t2m_cr_mean_10_12
timestamp_ccentral,,,,,,,
2015-12-31 18:00:00-06:00,0.0,t2m,265.051697,NaN,NaN,NaN,NaN
2015-12-31 19:00:00-06:00,0.0,t2m,265.200623,NaN,NaN,NaN,NaN
2015-12-31 20:00:00-06:00,0.0,t2m,265.428375,NaN,NaN,NaN,NaN
2015-12-31 21:00:00-06:00,0.0,t2m,266.728790,265.226898,NaN,NaN,NaN
2015-12-31 22:00:00-06:00,0.0,t2m,266.957642,265.785929,NaN,NaN,NaN


In [24]:
# Export datasets as csv (uncomment if need to re-export)
if not os.path.exists("t2m_cr_concentration_features.csv"):
    t2m_cr_final_features_conc.to_csv("t2m_cr_concentration_features.csv", index=True)
# t2m_cr_final_features_load.to_csv("t2m_cr_load_features_test1.csv", index=True)

Cypress River solar radiation features

In [25]:
cdir_cr_1_12_features = fc.calc_param_site_features_hrs_ago(df=full_dataset, 
                                                        param='cdir',
                                                        site='cr', 
                                                        hours_ago_start=1, 
                                                        hours_ago_end=12)

cdir_cr_13_48_features = fc.calc_param_site_features_hrs_ago(df=full_dataset, 
                                                        param='cdir',
                                                        site='cr', 
                                                        hours_ago_start=13, 
                                                        hours_ago_end=48)

cdir_cr_49_120_features = fc.calc_param_site_features_hrs_ago(df=full_dataset, 
                                                        param='cdir',
                                                        site='cr', 
                                                        hours_ago_start=49, 
                                                        hours_ago_end=120)

cdir_cr_121_720_features = fc.calc_param_site_features_hrs_ago(df=full_dataset, 
                                                        param='cdir',
                                                        site='cr', 
                                                        hours_ago_start=121, 
                                                        hours_ago_end=170)


# Create final turbidity concentration feature table (pnwa)
cdir_cr_final_features_conc = pd.concat([cdir_cr_1_12_features[0].set_index('timestamp_ccentral'),
                                        cdir_cr_13_48_features[0].set_index('timestamp_ccentral'),
                                        cdir_cr_49_120_features[0].set_index('timestamp_ccentral'),
                                        cdir_cr_121_720_features[0].set_index('timestamp_ccentral')
                                        ],
                                        axis=1)

# Create final turbidity load feature table (pnwa)
cdir_cr_final_features_load = pd.concat([cdir_cr_1_12_features[1].set_index('timestamp_ccentral'),
                                        cdir_cr_13_48_features[1].set_index('timestamp_ccentral'),
                                        cdir_cr_49_120_features[1].set_index('timestamp_ccentral'),
                                        cdir_cr_121_720_features[1].set_index('timestamp_ccentral')
                                        ],
                                        axis=1)

# Export datasets as csv (uncomment if need to re-export)
if not os.path.exists("cdir_cr_concentration_features.csv"):
    cdir_cr_final_features_conc.to_csv("cdir_cr_concentration_features.csv", index=True)
if not os.path.exists("cdir_cr_load_features.csv"):
    cdir_cr_final_features_load.to_csv("cdir_cr_load_features.csv", index=True)

Reviewing Cypress River solar radiation dataset

In [26]:
fc.feature_checks(cdir_cr_final_features_conc)
cdir_cr_final_features_conc.head()
#tur_pnwa_final_features_conc.dropna(axis = 0).head()

columns Index(['cdir_cr_mean_1_12', 'cdir_cr_mean_13_48', 'cdir_cr_mean_49_120',
       'cdir_cr_mean_121_170'],
      dtype='object')
start time 2015-12-31 19:00:00-06:00
end time 2020-12-31 17:00:00-06:00
number of rows 43848 and columns 4.000000
Count of non-null values by column


,cdir_cr_mean_1_12,cdir_cr_mean_13_48,cdir_cr_mean_49_120,cdir_cr_mean_121_170
timestamp_ccentral,,,,
2015-12-31 18:00:00-06:00,NaN,NaN,NaN,NaN
2015-12-31 19:00:00-06:00,NaN,NaN,NaN,NaN
2015-12-31 20:00:00-06:00,NaN,NaN,NaN,NaN
2015-12-31 21:00:00-06:00,NaN,NaN,NaN,NaN
2015-12-31 22:00:00-06:00,NaN,NaN,NaN,NaN


In [27]:
# Export datasets as csv (uncomment if need to re-export)
if not os.path.exists("cdir_cr_final_features.csv"):
    cdir_cr_final_features_conc.to_csv("cdir_cr_final_features.csv", index=True)

## Discharge features
Discharge lag feature generation

In [12]:
dschrg_cr_1_12_features = fc.calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='cr',
                                                        hours_ago_start=1,
                                                        hours_ago_end=12).set_index('timestamp_ccentral')
 
dschrg_cr_13_48_features = fc.calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='cr',
                                                        hours_ago_start=13,
                                                        hours_ago_end=48).set_index('timestamp_ccentral')
 
dschrg_cr_49_120_features = fc.calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='cr',
                                                        hours_ago_start=49,
                                                        hours_ago_end=120).set_index('timestamp_ccentral')
 
dschrg_cr_121_170_features = fc.calc_discharge_site_features_hrs_ago(df=full_dataset,
                                                        site='cr',
                                                        hours_ago_start=121,
                                                        hours_ago_end=170).set_index('timestamp_ccentral')
 

In [13]:
# Create discharge table to merge with time lagged features
full_df_central = full_dataset.iloc[:, 0:2].drop_duplicates() # drop duplicates (mostly due to dual t2m & cdir parameters)
full_df_central = full_df_central[~full_df_central['timestamp'].duplicated()] # drop duplicate timestamps
full_df_central['timestamp_ccentral'] = pd.to_datetime(full_dataset['timestamp'], utc=True).dt.tz_convert('Canada/Central')
full_df_central.drop('timestamp', axis=1, inplace=True)
full_df_central = full_df_central.set_index('timestamp_ccentral')


In [14]:
# Create the final discharge feature table
dschrg_cr_final_features = pd.concat([full_df_central,
                                      dschrg_cr_1_12_features,
                                        dschrg_cr_13_48_features,
                                        dschrg_cr_49_120_features,
                                        dschrg_cr_121_170_features],
                                          axis=1)


In [15]:
fc.feature_checks(dschrg_cr_final_features)
dschrg_cr_final_features.head()
dschrg_cr_final_features.dropna(axis = 0).head()

columns Index(['discharge', 'dschrg_cr_mean_1_12', 'dschrg_cr_mean_13_48',
       'dschrg_cr_mean_49_120', 'dschrg_cr_mean_121_170'],
      dtype='object')
start time 2015-12-31 19:00:00-06:00
end time 2020-12-31 17:00:00-06:00
number of rows 43848 and columns 5.000000
Count of non-null values by column


,discharge,dschrg_cr_mean_1_12,dschrg_cr_mean_13_48,dschrg_cr_mean_49_120,dschrg_cr_mean_121_170
timestamp_ccentral,,,,,
2016-01-07 20:00:00-06:00,0.0,0.0,0.0,0.0,0.0
2016-01-07 21:00:00-06:00,0.0,0.0,0.0,0.0,0.0
2016-01-07 22:00:00-06:00,0.0,0.0,0.0,0.0,0.0
2016-01-07 23:00:00-06:00,0.0,0.0,0.0,0.0,0.0
2016-01-08 00:00:00-06:00,0.0,0.0,0.0,0.0,0.0


In [16]:
# # Export dataset as csv (uncomment if need to re-export)
dschrg_cr_final_features.to_csv("discharge_cr_features_2016-2020.csv", index=True)


## 2. Combine Previous Feature Sets Generated

In [21]:
from feature_creation import read_and_merge_csv

#All the t2m and cdir files
t2m_files = [
    't2m_cr_concentration_features.csv',
    't2m_cr_load_features.csv'
]

cdir_files = [
    'cdir_cr_concentration_features.csv',
    'cdir_cr_load_features.csv'
]

discharge_files = [
    'discharge_cr_features_2016-2020.csv'
]

#Combine all t2m files
t2m_combined_df = read_and_merge_csv(t2m_files, '')

#Combine all cdir files
cdir_combined_df = read_and_merge_csv(cdir_files, '')

#Combine all discharge files
discharge_combined_df = read_and_merge_csv(discharge_files, '')

#Combine the discharge data with t2m and cdir combined data
t2m_merged_df = pd.merge(t2m_combined_df, discharge_combined_df, on="timestamp_ccentral", how="outer")
cdir_merged_df = pd.merge(cdir_combined_df, discharge_combined_df, on="timestamp_ccentral", how="outer")

#Save the combined csv files
t2m_merged_df.to_csv(os.path.join('t2m_input_features.csv'), index=False)
cdir_merged_df.to_csv(os.path.join('cdir_input_features.csv'), index=False)


In [23]:

# ## Refer to files:
# ## 1. 3h_toc_input_features.csv
# ## 2. 3h_tur_input_features.csv

# # Glimpse the combined feature sets
# toc_merged_df = pd.read_csv(os.path.join(project_conf.PROCESSED_DATA_DIR, '3h_toc_input_features.csv'))
# tur_merged_df = pd.read_csv(os.path.join(project_conf.PROCESSED_DATA_DIR, '3h_tur_input_features.csv'))

# view_toc.head()
# # view_tur.head()


TP data added manually